# Laboration 1

## Excercise 1

In [1]:
import torch

a = torch.tensor([2.0, 0.0, 0.0])
b = torch.tensor([1.0, 0.25, 0.0])
c = torch.tensor([1.0, 1.0, 1.0])

V = a @ (torch.linalg.cross(b, c))
print(V)

tensor(0.5000)


## Excercise 2

In [2]:
q = torch.tensor([[1, 2], [3, 4], [5, 6]])
a = torch.tensor([1, 0, 1])

print(q[torch.arange(q.shape[0]), a])

tensor([2, 3, 6])


## Excercise 3

In [3]:
ALPHA = 1e-1
ITERATIONS = 100
xyz = torch.tensor([1.0, 1.0, 1.0], dtype = torch.float32, requires_grad=True)

for _ in range(ITERATIONS):
    x, y, z = xyz
    f = x**2 - 5*x + y**2 - y + z**2 - 3*z + 8.75
    f.backward()

    with torch.no_grad():
        xyz -= ALPHA * xyz.grad
        xyz.grad.zero_()

print(f"Final xyz: {xyz.detach()}")

Final xyz: tensor([2.5000, 0.5000, 1.5000])


## Excercise 4

In [4]:
import torch.nn as nn


class poly(nn.Module):

    def __init__(self):
        super().__init__()
        self.C = nn.Parameter(torch.tensor([1.0, 2.0, 3.0]))
        self.powers = nn.Parameter(torch.tensor([0.0, 1.0, 2.0]))

    def forward(self, x):
        x = torch.cat([x, x, x], dim=1) ** self.powers
        y = torch.einsum("n, n... -> ...", self.C, x.T)
        return y[:, None]


polynomial = poly()

optimizer = torch.optim.Adam(polynomial.parameters(), lr=1e-1)

test_x = torch.tensor([[0.0987], [0.5131], [1.1002], [1.6118], [1.8777]])
test_y = torch.tensor([[1.2265], [2.8162], [6.8318], [12.0178], [15.3323]])

for _ in range(500):
    y_guess = polynomial(test_x)
    loss = torch.nn.functional.mse_loss(input=y_guess, target=test_y)
    loss.backward()

    optimizer.step()
    optimizer.zero_grad()

print(polynomial.C.detach())

tensor([0.9306, 1.9399, 3.1296])


## Excercise 5

In [5]:
def greedy_actions(net: nn.Module, states: torch.Tensor):
    "Takes batches of states and returns the action for each state."

    action_probabilities = net(states)
    actions = torch.argmax(action_probabilities, dim=1)
    return actions


Qnet = nn.Sequential(nn.Linear(10, 20), nn.ReLU(),
                     nn.Linear(20, 40), nn.ReLU(),
                     nn.Linear(40, 20), nn.ReLU(),
                     nn.Linear(20, 6))

# Example

test_states = torch.rand([32, 10])
test_actions = greedy_actions(Qnet, test_states)

print(f"Size of input states: {test_states.shape}")
print(f"Size of output actions: {test_actions.shape}")

Size of input states: torch.Size([32, 10])
Size of output actions: torch.Size([32])


## Excercise 6

In [6]:
from enum import Enum
import gymnasium as gym
from gymnasium import spaces
import pygame
import numpy as np


class GridWorldEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 24}

    SQUARE_PX = 240
    GROUND_COLOR = 0x969696
    OBSTACLE_COLOR = 0
    TERMINAL_COLOR = 0xFFFFFF
    AGENT_COLOR = 0xC80000

    def __init__(self, world, terminal_dict, action_map, start,
                 frame_rate=24, render_mode=None):
        super().__init__()
        self.world = np.asarray(world)
        self.env_height, self.env_width = self.world.shape
        self.terminal_states = dict(terminal_dict)
        self.action_map = dict(action_map)
        self.start_state = np.asarray(start, dtype=int).copy()

        self._terminal_mask = np.zeros(self.world.shape, dtype=bool)
        for (x, y) in self.terminal_states:
            self._terminal_mask[y, x] = True
        self._target_locations = np.argwhere(self._terminal_mask)[:, ::-1]  # (x, y)

        self.action_space = spaces.Discrete(len(self.action_map))
        self.observation_space = spaces.Dict({
            "agent": spaces.MultiDiscrete(self.world.shape[::-1])
        })

        self.metadata = {**self.metadata, "render_fps": frame_rate}
        assert render_mode in (None, *self.metadata["render_modes"])
        self.render_mode = render_mode

        self._agent_location = self.start_state.copy()
        self.total_reward = 0.0
        self.window = self.clock = self._static_surface = None
        self._min = np.zeros(2, dtype=int)
        self._max = np.array([self.env_width, self.env_height]) - 1

    def _is_obstacle(self, x, y):    return self.world[y, x] == 1
    def _is_terminal(self, x, y):   return self._terminal_mask[y, x]

    def _get_obs(self):
        return {"agent": self._agent_location}

    def _get_info(self):
        return {
            "total reward": self.total_reward,
        }

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.total_reward = 0.0
        self._agent_location = self.start_state.copy()
        obs, info = self._get_obs(), self._get_info()
        if self.render_mode == "human":
            self._render_frame()
        return obs, info

    def step(self, action):
        direction = self.action_map[action]
        new_loc = np.clip(self._agent_location + direction, self._min, self._max)
        if not self._is_obstacle(*new_loc):
            self._agent_location = new_loc
        terminated = self._is_terminal(*self._agent_location)
        reward = float(self.terminal_states[(self._agent_location[0], self._agent_location[1])]) if terminated else 0.0
        self.total_reward += reward
        if self.render_mode == "human":
            self._render_frame()
        return self._get_obs(), reward, terminated, False, self._get_info()

    def render(self):
        if self.render_mode == "rgb_array":
            return self._render_frame()

    def _render_frame(self):
        square_pixel_size = 240
        ground_color: int = 0x969696
        obstacle_color: int = 0
        terminal_color: int = 0xFFFFFF
        agent_color: int = 0xC80000

        square_pixel_dims = [square_pixel_size, square_pixel_size]
        win_width = self.env_width * square_pixel_size
        win_height = self.env_height * square_pixel_size
        env_height = self.env_height
        env_width = self.env_width

        if self.window is None and self.render_mode == "human":
            pygame.init()
            pygame.display.init()
            self.window = pygame.display.set_mode((win_width, win_height))
            pygame.display.set_caption("GridWorld")
        if self.clock is None and self.render_mode == "human":
            self.clock = pygame.time.Clock()

        canvas = pygame.Surface((win_width, win_height))
        canvas.fill(ground_color)

        # Draw obstacles and grid
        for y in range(env_height):
            pygame.draw.line(
                canvas,
                obstacle_color,
                (0, square_pixel_size * y),
                (win_width, square_pixel_size * y),
                width=3,
            )
            for x in range(env_width):
                if y == 0:
                    pygame.draw.line(
                        canvas,
                        obstacle_color,
                        (square_pixel_size * x, 0),
                        (square_pixel_size * x, win_height),
                        width=3,
                    )
                if self._is_obstacle(x, y):
                    loc = np.array([x, y])
                    pygame.draw.rect(
                        canvas,
                        obstacle_color,
                        pygame.Rect(square_pixel_dims * loc, square_pixel_dims),
                    )
                elif self._is_terminal(x, y):
                    loc = np.array([x, y])
                    pygame.draw.rect(
                        canvas,
                        terminal_color,
                        pygame.Rect(square_pixel_dims * loc, square_pixel_dims),
                    )

        # Draw agent
        pygame.draw.rect(
            canvas,
            agent_color,
            pygame.Rect(square_pixel_dims * self._agent_location, square_pixel_dims),
        )

        if self.render_mode == "human":
            self.window.blit(canvas, canvas.get_rect())
            pygame.event.pump()
            pygame.display.update()
            self.clock.tick(self.metadata["render_fps"])
        else:
            return np.transpose(
                np.array(pygame.surfarray.pixels3d(canvas)), axes=(1, 0, 2)
            )

    def close(self):
        if self.window is not None:
            pygame.display.quit()
            pygame.quit()


In [7]:
from gymnasium.envs.registration import register

register(
    id="GridWorldEnv",
    entry_point=GridWorldEnv
)

In [8]:
world = np.array([[1, 0, 1], [1, 0, 1], [1, 0, 1], [0, 0, 0]])

env = gym.make(
    "GridWorldEnv",
    world=world,
    action_map={
        0: np.array([0, -1]),  # Up
        1: np.array([1, 0]),  # Right
        2: np.array([0, 1]),  # Down
        3: np.array([-1, 0]),  # Left
    },
    start=np.array([1, 0]),
    terminal_dict={
        (0, 3): -1,
        (2, 3): 1,
    },  # (key,value)-pairs by terminal state: reward
    render_mode="human",
    frame_rate=48,
    max_episode_steps=100,
)

for episode in range(5):
    env.reset()
    while True:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        if terminated or truncated:
            tot_rew = info["total reward"]
            print(f"Episode {episode} total reward: {tot_rew}")
            break

env.close()

c:\software\gitlibraries\Schoolwork\Python\DeepReinforcementLearning\.DRLenv\Lib\site-packages\gymnasium\utils\passive_env_checker.py:237: UserWarning: WARN: The observations returned by `reset` and the following `step` share an object, therefore, modifying one will modify the other. Environments should return new observation data on every call as users commonly keep the data returned to them.
  logger.warn(


Episode 0 total reward: -1.0
Episode 1 total reward: 1.0
Episode 2 total reward: 1.0
Episode 3 total reward: 1.0
Episode 4 total reward: -1.0


## Excercise 7

In [9]:
import random
from collections import namedtuple, deque

Transition = namedtuple(
    "Transition", ("state", "action", "reward", "next_state", "terminated")
)
Batch = namedtuple(
    "Batch", ("states", "actions", "rewards", "next_states", "terminateds")
)

class ReplayMemory:

    def __init__(self, capacity: int, batch_size: int):
        self.batch_size = batch_size
        self.memory = deque([], maxlen=capacity)

    def add(self, *args):
        self.memory.append(Transition(*args))

    def sample(self) -> Batch:
        sample = random.choices(self.memory, k=self.batch_size)
        states, actions, rewards, next_states, terminateds = list(zip(*sample))
        states = torch.tensor(np.array(states), dtype=torch.float32)
        actions = torch.tensor(np.array(actions), dtype=torch.int64)
        rewards = torch.tensor(np.array(rewards), dtype=torch.float32)
        next_states = torch.tensor(np.array(next_states), dtype=torch.float32)
        terminateds = torch.tensor(np.array(terminateds), dtype=torch.float32)

        return Batch(states, actions, rewards, next_states, terminateds)

In [ ]:
class Net(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.hidden = hidden
        self.layers = nn.ModuleList(
            [nn.Linear(hidden[i-1], hidden[i]) for i in range(1, len(hidden))]
        )

    def forward(self, x):
        assert x.shape[-1] == self.hidden[0]
        y = x
        for j, layer in enumerate(self.layers):
            y = layer(y)
            if j < len(self.layers) - 1:
                y = nn.functional.relu(y)
        return y


class Agent:
    def __init__(self, *hidden, gamma, lr):
        self.Qnet = Net(hidden)
        self.gamma = gamma
        self.optimizer = torch.optim.Adam(self.Qnet.parameters(), lr=lr)

    @torch.no_grad()
    def greedy_action(self, state):
        q = self.Qnet(state)
        a = q.argmax(dim=0)
        return a.item() if a.numel() == 1 else a

    @torch.no_grad
    def update(self, state, action, reward, next_state, terminated) -> float:
        q_sa = self.Qnet(state).gather(1, action.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            next_q_max = self.Qnet(next_state).max(dim=1).values
            # zero the bootstrap on terminal transitions
            not_done = 1.0 - terminated.float()
            target = reward + self.gamma * not_done * next_q_max

        # ---- 3. TD(0) loss on the taken action only ----
        loss = nn.functional.mse_loss(q_sa, target)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item()

    def batch_update(self, states, actions, rewards, next_states, terminated) -> float:

        q_sa = self.Qnet(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            next_q_max = self.Qnet(next_states).max(dim=1).values
            not_done   = 1.0 - terminated.to(q_sa.dtype)
            target     = rewards + self.gamma * not_done * next_q_max

        loss = nn.functional.mse_loss(q_sa, target)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item()

In [15]:
import pandas as pd 

NUM_EXPERIMENTS = 20
NUM_EPISODES = 25
GAMMA = 0.99
LR = 1e-3
TAU = lambda n : 0.2 ** (2 / n)
EPS_INIT = 1.0
EPS_MIN = 1e-2
MAX_EPISODE_STEPS = 1000
BUFFER_SIZE = 1000
BATCH_SIZE = 32

WORLD = np.array([[0, 1, 0, 0, 0], [0, 0, 0, 1, 0]])
ACTIONS = {
        0: np.array([0, -1]),  # Up
        1: np.array([1, 0]),  # Right
        2: np.array([0, 1]),  # Down
        3: np.array([-1, 0]),  # Left
    }
START = np.array([0, 1])
TERMINAL_DICT = {
        (0, 0): 0.5,
        (4, 1): 1.0,
    }


def experiment_qlearn(env) -> pd.DataFrame:
    dfs = []
    for exp in range(NUM_EXPERIMENTS):
        print(f"QLR Running experiment {exp}", end="\r")

        returns = []
        model = Agent(2, 10, 4, gamma=GAMMA, lr=LR)

        for episode in range(NUM_EPISODES):
            state = torch.tensor(START, dtype=torch.float32)
            env.reset()
            while True:
                action = model.greedy_action(state)
                obs, reward, terminated, truncated, info = env.step(action)
                model.update(state, action, reward, obs["agent"], terminated)   # Learning
                state = obs["agent"]
                if terminated or truncated:
                    tot_rew = info["total reward"]
                    returns.append(tot_rew)
                    break

        dfs.append(
            pd.DataFrame(
                {"exp": exp, "episode": np.arange(NUM_EPISODES), "return": returns}
            )
        )

    df = pd.concat(dfs, ignore_index=True)
    df["algo"] = "q-learn"
    print(df)
    return df

env = gym.make(
    "GridWorldEnv",
    world=WORLD,
    action_map=ACTIONS,
    start=START,
    terminal_dict=TERMINAL_DICT,
    render_mode="rgb_array",
    frame_rate=120,
    max_episode_steps=MAX_EPISODE_STEPS
)

experiment_qlearn(env)

env.close()

IndexError: Dimension out of range (expected to be in range of [-1, 0], but got 1)